# Task
Load the data from "comprehensive_dataset_final.parquet" located in Google Drive, preprocessing it by removing rows with missing 'Bid ask spread'. Train Elastic Net and Tree-based models to predict 'Yield to maturity' (excluding 'Bid ask spread' and 'days since trade' for the Elastic Net model) and 'Bid ask spread', then evaluate the performance of these models.

## Mount Google Drive

### Subtask:
Mount Google Drive to access the file system.


**Reasoning**:
Import the `drive` module from `google.colab` and mount Google Drive at `/content/drive` to access files.



In [ ]:
#==============================================================================
# CELL 1: SETUP AND MOUNT GOOGLE DRIVE
#==============================================================================

"""
ELASTIC NET FOR YTM AND BID-ASK SPREAD PREDICTION
==================================================
Complete self-contained notebook that:
1. Loads data from Google Drive
2. Preprocesses (winsorizing, handling missing values, etc.)
3. Creates temporal train/val/test splits
4. Runs parallelized Elastic Net for both YTM and Bid-Ask Spread

Expected runtime: ~10-15 minutes total
"""

from google.colab import drive
drive.mount('/content/drive')

#==============================================================================
# CELL 2: IMPORTS
#==============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
from joblib import Parallel, delayed
import gc
import time
import os
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("ELASTIC NET FOR YTM AND BID-ASK SPREAD PREDICTION")
print("="*80)

n_cores = os.cpu_count()
print(f"\nAvailable CPU cores: {n_cores}")

#==============================================================================
# CELL 3: LOAD DATA
#==============================================================================

print("\n" + "="*80)
print("STEP 1: LOADING DATA")
print("="*80)

# UPDATE THIS PATH TO MATCH YOUR GOOGLE DRIVE LOCATION
DATA_PATH = '/content/drive/MyDrive/comprehensive_dataset_final.parquet'

print(f"\nLoading data from: {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)

print(f"   Shape: {df.shape}")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"   Columns: {len(df.columns)}")

print(f"\nDataset Overview:")
print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
print(f"   Unique bonds (CUSIPs): {df['cusip'].nunique():,}")

# Check target variables
print(f"\nTarget Variables:")
print(f"   YTM: {df['ytm'].notna().sum():,} non-null ({df['ytm'].notna().mean()*100:.1f}%)")
print(f"   Bid-Ask Spread: {df['bid_ask_spread_20d'].notna().sum():,} non-null ({df['bid_ask_spread_20d'].notna().mean()*100:.1f}%)")

#==============================================================================
# CELL 4: PREPROCESSING
#==============================================================================

print("\n" + "="*80)
print("STEP 2: DATA PREPROCESSING")
print("="*80)

# Ensure date is datetime and sorted
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Define column categories
id_cols = ['cusip', 'issuer_cusip', 'date']
target_cols = ['ytm', 'bid_ask_spread_20d', 'ret_eom']

# Leakage features to exclude (contain information derived from target)
LEAKAGE_FEATURES = [
    'ytm_spread_to_10y',
    'ytm_spread_to_ig',
    'ytm_spread_to_comparable',
    'ytm_rank',
    'coupon_to_ytm',
]

# Identify feature columns
exclude_cols = id_cols + target_cols + LEAKAGE_FEATURES
all_features = [c for c in df.columns if c not in exclude_cols]
numeric_features = df[all_features].select_dtypes(include=[np.number]).columns.tolist()

print(f"\nFeature Summary:")
print(f"   Total columns: {len(df.columns)}")
print(f"   ID columns: {len(id_cols)}")
print(f"   Target columns: {len(target_cols)}")
print(f"   Leakage features removed: {len(LEAKAGE_FEATURES)}")
print(f"   Numeric features for modeling: {len(numeric_features)}")

#==============================================================================
# CELL 5: TIME-BASED TRAIN/VAL/TEST SPLIT
#==============================================================================

print("\n" + "="*80)
print("STEP 3: CREATING TRAIN/VALIDATION/TEST SPLITS")
print("="*80)

# Time-based splits following Gu, Kelly, Xiu (2020) methodology
TRAIN_END = '2021-12-31'
VAL_END = '2023-06-30'

print(f"\nTime-based splitting:")
print(f"   Training:   up to {TRAIN_END}")
print(f"   Validation: {TRAIN_END} to {VAL_END}")
print(f"   Test:       after {VAL_END}")

# Create masks
train_mask = df['date'] <= TRAIN_END
val_mask = (df['date'] > TRAIN_END) & (df['date'] <= VAL_END)
test_mask = df['date'] > VAL_END

df_train_full = df[train_mask].copy()
df_val_full = df[val_mask].copy()
df_test_full = df[test_mask].copy()

print(f"\nFull dataset split sizes:")
print(f"   Train: {len(df_train_full):,}")
print(f"   Val:   {len(df_val_full):,}")
print(f"   Test:  {len(df_test_full):,}")

# Free memory
del df
gc.collect()

#==============================================================================
# CELL 6: HELPER FUNCTIONS
#==============================================================================

def evaluate_enet_params(alpha, l1_ratio, X_train, y_train, X_val, y_val):
    """Evaluate a single Elastic Net parameter combination."""
    model = SGDRegressor(
        loss='squared_error',
        penalty='elasticnet',
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=1000,
        tol=1e-4,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        random_state=42,
        learning_rate='adaptive',
        eta0=0.01
    )
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    return {'alpha': alpha, 'l1_ratio': l1_ratio, 'rmse': rmse}


def preprocess_for_elastic_net(train_df, val_df, test_df, features, target_col,
                                winsorize=True, winsorize_lower=0.001, winsorize_upper=0.999):
    """
    Complete preprocessing pipeline for Elastic Net:
    1. Drop rows with missing target
    2. Winsorize target (optional)
    3. Handle inf values in features
    4. Remove zero-variance columns
    5. Fill NaN and standardize

    Returns: X_train, X_val, X_test, y_train, y_val, y_test, valid_features, scaler
    """

    # Step 1: Drop rows with missing target
    train_clean = train_df.dropna(subset=[target_col]).copy()
    val_clean = val_df.dropna(subset=[target_col]).copy()
    test_clean = test_df.dropna(subset=[target_col]).copy()

    print(f"   Samples with valid {target_col}:")
    print(f"      Train: {len(train_clean):,}, Val: {len(val_clean):,}, Test: {len(test_clean):,}")

    # Step 2: Winsorize target variable (using training quantiles only)
    if winsorize:
        lower = train_clean[target_col].quantile(winsorize_lower)
        upper = train_clean[target_col].quantile(winsorize_upper)

        print(f"   Winsorizing {target_col} at [{lower:.6f}, {upper:.6f}]")

        for df in [train_clean, val_clean, test_clean]:
            df[target_col] = df[target_col].clip(lower, upper)

    # Step 3: Handle inf values in features
    for df in [train_clean, val_clean, test_clean]:
        df[features] = df[features].replace([np.inf, -np.inf], np.nan)

    # Step 4: Remove zero-variance columns (based on training data)
    variance = train_clean[features].var()
    valid_features = [f for f in features if variance.get(f, 0) >= 1e-10]

    removed_cols = len(features) - len(valid_features)
    if removed_cols > 0:
        print(f"   Removed {removed_cols} zero/low variance columns")

    print(f"   Final features: {len(valid_features)}")

    # Step 5: Fill NaN with 0 and standardize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_clean[valid_features].fillna(0))
    X_val = scaler.transform(val_clean[valid_features].fillna(0))
    X_test = scaler.transform(test_clean[valid_features].fillna(0))

    # Safety check: replace any remaining inf/nan
    X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
    X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
    X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

    # Extract target values
    y_train = train_clean[target_col].values
    y_val = val_clean[target_col].values
    y_test = test_clean[target_col].values

    print(f"   X_train shape: {X_train.shape}")
    print(f"   Any inf remaining: {np.isinf(X_train).any()}")
    print(f"   Any NaN remaining: {np.isnan(X_train).any()}")

    return X_train, X_val, X_test, y_train, y_val, y_test, valid_features, scaler


def run_elastic_net(X_train, y_train, X_val, y_val, X_test, y_test,
                    feature_names, target_name, subsample_frac=0.1):
    """
    Run parallelized Elastic Net hyperparameter tuning and training.

    Returns: model, best_params, metrics, predictions
    """
    print(f"\n{'='*60}")
    print(f"ELASTIC NET FOR {target_name.upper()}")
    print(f"{'='*60}")

    start_time = time.time()

    # Subsample for faster tuning
    np.random.seed(42)
    n_sample = int(subsample_frac * len(X_train))
    sample_idx = np.random.choice(len(X_train), n_sample, replace=False)
    X_train_sample = X_train[sample_idx]
    y_train_sample = y_train[sample_idx]

    print(f"\n1. Hyperparameter Tuning (Parallelized)")
    print(f"   Training samples: {len(X_train):,}")
    print(f"   Tuning subsample: {n_sample:,} ({subsample_frac*100:.0f}%)")

    # Parameter grid
    param_grid = {
    'alpha': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'l1_ratio': [0.0]  # Pure Ridge, no feature selection
}

    param_combinations = [
        (alpha, l1)
        for alpha in param_grid['alpha']
        for l1 in param_grid['l1_ratio']
    ]

    print(f"   Testing {len(param_combinations)} combinations in parallel...")

    # Parallel grid search
    results = Parallel(n_jobs=-1, verbose=1)(
        delayed(evaluate_enet_params)(
            alpha, l1, X_train_sample, y_train_sample, X_val, y_val
        )
        for alpha, l1 in param_combinations
    )

    # Find best (filter out any extreme RMSE values first)
    valid_results = [r for r in results if r['rmse'] < 1e6]
    if not valid_results:
        print("   WARNING: All results have very high RMSE. Using least bad option.")
        valid_results = results

    best_result = min(valid_results, key=lambda x: x['rmse'])
    best_params = {'alpha': best_result['alpha'], 'l1_ratio': best_result['l1_ratio']}

    tuning_time = time.time() - start_time
    print(f"\n   Tuning completed in {tuning_time:.1f} seconds")

    # Print top 5 results
    print(f"\n   Top 5 Results (by RMSE):")
    for i, r in enumerate(sorted(valid_results, key=lambda x: x['rmse'])[:5], 1):
        marker = " <-- BEST" if r == best_result else ""
        print(f"      [{i}] alpha={r['alpha']:.0e}, l1={r['l1_ratio']:.2f}, RMSE={r['rmse']:.6f}{marker}")

    print(f"\n   Best: alpha={best_params['alpha']:.0e}, l1_ratio={best_params['l1_ratio']:.2f}")

    # Train final model on full data
    print(f"\n2. Training Final Model on Full Data...")
    train_start = time.time()

    final_model = SGDRegressor(
        loss='squared_error',
        penalty='elasticnet',
        alpha=best_params['alpha'],
        l1_ratio=best_params['l1_ratio'],
        max_iter=2000,
        tol=1e-5,
        early_stopping=False,
        random_state=42,
        learning_rate='adaptive',
        eta0=0.01
    )

    final_model.fit(X_train, y_train)

    print(f"   Training completed in {time.time() - train_start:.1f} seconds")

    # Predictions
    y_train_pred = final_model.predict(X_train)
    y_val_pred = final_model.predict(X_val)
    y_test_pred = final_model.predict(X_test)

    # Metrics
    metrics = {
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_train_pred)),
        'val_rmse': np.sqrt(mean_squared_error(y_val, y_val_pred)),
        'test_rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
        'train_r2': r2_score(y_train, y_train_pred),
        'val_r2': r2_score(y_val, y_val_pred),
        'test_r2': r2_score(y_test, y_test_pred),
    }

    n_nonzero = (final_model.coef_ != 0).sum()

    print(f"\n3. Performance:")
    print(f"   Train RMSE: {metrics['train_rmse']:.6f}, R²: {metrics['train_r2']:.4f}")
    print(f"   Val RMSE:   {metrics['val_rmse']:.6f}, R²: {metrics['val_r2']:.4f}")
    print(f"   Test RMSE:  {metrics['test_rmse']:.6f}, R²: {metrics['test_r2']:.4f}")
    print(f"   Non-zero features: {n_nonzero}/{len(feature_names)}")

    total_time = time.time() - start_time
    print(f"\n   Total time: {total_time/60:.1f} minutes")

    return final_model, best_params, metrics, y_test_pred


#==============================================================================
# CELL 7: ELASTIC NET FOR YTM
#==============================================================================

print("\n" + "="*80)
print("PART A: YIELD TO MATURITY (YTM)")
print("="*80)

print("\nPreprocessing for YTM prediction...")

X_train_ytm, X_val_ytm, X_test_ytm, y_train_ytm, y_val_ytm, y_test_ytm, features_ytm, scaler_ytm = \
    preprocess_for_elastic_net(
        df_train_full, df_val_full, df_test_full,
        numeric_features,
        target_col='ytm',
        winsorize=True,
        winsorize_lower=0.001,
        winsorize_upper=0.999
    )

# Run Elastic Net
ytm_model, ytm_params, ytm_metrics, ytm_pred = run_elastic_net(
    X_train_ytm, y_train_ytm, X_val_ytm, y_val_ytm, X_test_ytm, y_test_ytm,
    features_ytm, 'YTM'
)

# Feature importance for YTM
print("\n4. Top 15 Features by |Coefficient|:")
coef_df_ytm = pd.DataFrame({
    'feature': features_ytm,
    'coefficient': ytm_model.coef_
})
coef_df_ytm['abs_coef'] = np.abs(coef_df_ytm['coefficient'])
coef_df_ytm = coef_df_ytm.sort_values('abs_coef', ascending=False)

for i, row in coef_df_ytm.head(15).iterrows():
    sign = "+" if row['coefficient'] > 0 else ""
    print(f"   {sign}{row['coefficient']:+.4f}  {row['feature']}")

# Clean up
del X_train_ytm, X_val_ytm, X_test_ytm
gc.collect()

#==============================================================================
# CELL 8: ELASTIC NET FOR BID-ASK SPREAD
#==============================================================================

print("\n" + "="*80)
print("PART B: BID-ASK SPREAD")
print("="*80)

print("\nPreprocessing for Bid-Ask Spread prediction...")

X_train_spread, X_val_spread, X_test_spread, y_train_spread, y_val_spread, y_test_spread, features_spread, scaler_spread = \
    preprocess_for_elastic_net(
        df_train_full, df_val_full, df_test_full,
        numeric_features,
        target_col='bid_ask_spread_20d',
        winsorize=True,
        winsorize_lower=0.001,
        winsorize_upper=0.999
    )

# Run Elastic Net
spread_model, spread_params, spread_metrics, spread_pred = run_elastic_net(
    X_train_spread, y_train_spread, X_val_spread, y_val_spread, X_test_spread, y_test_spread,
    features_spread, 'BID-ASK SPREAD'
)

# Feature importance for Spread
print("\n4. Top 15 Features by |Coefficient|:")
coef_df_spread = pd.DataFrame({
    'feature': features_spread,
    'coefficient': spread_model.coef_
})
coef_df_spread['abs_coef'] = np.abs(coef_df_spread['coefficient'])
coef_df_spread = coef_df_spread.sort_values('abs_coef', ascending=False)

for i, row in coef_df_spread.head(15).iterrows():
    sign = "+" if row['coefficient'] > 0 else ""
    print(f"   {sign}{row['coefficient']:+.4f}  {row['feature']}")

#==============================================================================
# CELL 9: SAVE RESULTS
#==============================================================================

print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Save coefficients
coef_df_ytm.to_csv('enet_ytm_coefficients.csv', index=False)
coef_df_spread.to_csv('enet_spread_coefficients.csv', index=False)

print("\nSaved coefficient files:")
print("   enet_ytm_coefficients.csv")
print("   enet_spread_coefficients.csv")

#==============================================================================
# CELL 10: FINAL SUMMARY
#==============================================================================

print("\n" + "="*80)
print("ELASTIC NET SUMMARY")
print("="*80)

print(f"""
{'':40} {'YTM':>15} {'Bid-Ask Spread':>15}
{'-'*70}
Best Alpha:                              {ytm_params['alpha']:<15.0e} {spread_params['alpha']:<15.0e}
Best L1 Ratio:                           {ytm_params['l1_ratio']:<15.2f} {spread_params['l1_ratio']:<15.2f}

Train RMSE:                              {ytm_metrics['train_rmse']:<15.6f} {spread_metrics['train_rmse']:<15.6f}
Val RMSE:                                {ytm_metrics['val_rmse']:<15.6f} {spread_metrics['val_rmse']:<15.6f}
Test RMSE:                               {ytm_metrics['test_rmse']:<15.6f} {spread_metrics['test_rmse']:<15.6f}

Train R²:                                {ytm_metrics['train_r2']:<15.4f} {spread_metrics['train_r2']:<15.4f}
Val R²:                                  {ytm_metrics['val_r2']:<15.4f} {spread_metrics['val_r2']:<15.4f}
Test R²:                                 {ytm_metrics['test_r2']:<15.4f} {spread_metrics['test_r2']:<15.4f}

Non-zero Features:                       {(ytm_model.coef_ != 0).sum()}/{len(features_ytm):<10} {(spread_model.coef_ != 0).sum()}/{len(features_spread):<10}
""")

print("\nNote: These parquet files are also saved for use in the LightGBM notebook:")
print("   tree_ytm_train.parquet, tree_ytm_val.parquet, tree_ytm_test.parquet")
print("   enet_ytm_train.parquet, enet_ytm_val.parquet, enet_ytm_test.parquet")

print("\n" + "="*80)
print("ELASTIC NET COMPLETE")
print("="*80)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ELASTIC NET FOR YTM AND BID-ASK SPREAD PREDICTION

Available CPU cores: 8

STEP 1: LOADING DATA

Loading data from: /content/drive/MyDrive/comprehensive_dataset_final.parquet
   Shape: (2347177, 241)
   Memory: 5.49 GB
   Columns: 241

Dataset Overview:
   Date range: 2015-01-31 00:00:00 to 2024-08-31 00:00:00
   Unique bonds (CUSIPs): 78,421

Target Variables:
   YTM: 2,347,177 non-null (100.0%)
   Bid-Ask Spread: 1,556,420 non-null (66.3%)

STEP 2: DATA PREPROCESSING

Feature Summary:
   Total columns: 241
   ID columns: 3
   Target columns: 3
   Leakage features removed: 5
   Numeric features for modeling: 216

STEP 3: CREATING TRAIN/VALIDATION/TEST SPLITS

Time-based splitting:
   Training:   up to 2021-12-31
   Validation: 2021-12-31 to 2023-06-30
   Test:       after 2023-06-30

Full dataset split sizes:
   Train: 1,612,439
   Val:   473,211
   Test:  2

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:   11.3s remaining:   17.0s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   57.8s finished



   Tuning completed in 58.2 seconds

   Top 5 Results (by RMSE):
      [1] alpha=1e+00, l1=0.00, RMSE=2.424196 <-- BEST

   Best: alpha=1e+00, l1_ratio=0.00

2. Training Final Model on Full Data...
   Training completed in 148.2 seconds

3. Performance:
   Train RMSE: 2.076534, R²: 0.4413
   Val RMSE:   2.293658, R²: -0.2310
   Test RMSE:  2.109085, R²: -0.1074
   Non-zero features: 215/215

   Total time: 3.4 minutes

4. Top 15 Features by |Coefficient|:
   ++0.3865  ytm_lag6m
   ++0.2646  distance_to_ig
   ++0.2420  ytm_lag3m
   ++0.1550  realized_vol_6m
   -0.1444  price_to_ma_12m
   ++0.1424  realized_vol_3m
   ++0.1387  return_vol_12m
   -0.1366  ig_indicator
   ++0.1301  lagged_spread
   ++0.1249  return_vol_6m
   -0.1160  return_12m
   ++0.1109  rating_num
   -0.1070  price_to_ma_6m
   ++0.1003  zero_volume_3m
   ++0.0951  spread_volatility_3m

PART B: BID-ASK SPREAD

Preprocessing for Bid-Ask Spread prediction...
   Samples with valid bid_ask_spread_20d:
      Train: 973,435, 

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of   5 | elapsed:    8.2s remaining:   12.3s
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:  2.0min finished



   Tuning completed in 119.0 seconds

   Top 5 Results (by RMSE):
      [1] alpha=1e+00, l1=0.00, RMSE=0.013707 <-- BEST

   Best: alpha=1e+00, l1_ratio=0.00

2. Training Final Model on Full Data...
   Training completed in 56.9 seconds

3. Performance:
   Train RMSE: 0.014965, R²: 0.0583
   Val RMSE:   0.013336, R²: -0.0068
   Test RMSE:  0.012497, R²: 0.0176
   Non-zero features: 213/213

   Total time: 2.9 minutes

4. Top 15 Features by |Coefficient|:
   ++0.0007  spread_rank
   ++0.0006  discount_bond
   -0.0005  log_issue_size
   ++0.0004  zero_volume_3m
   ++0.0004  avg_spread_monthly
   ++0.0003  avg_spread_3m
   ++0.0003  vix_rating_interaction
   ++0.0003  spread_volatility_3m
   ++0.0003  ytm_lag6m
   ++0.0003  missing_ytm
   ++0.0003  lagged_spread
   ++0.0002  volume_monthly
   ++0.0002  avg_spread_monthly_lag6m
   ++0.0002  zero_volume
   -0.0002  high_coupon

SAVING RESULTS

Saved coefficient files:
   enet_ytm_coefficients.csv
   enet_spread_coefficients.csv

ELASTIC NE

In [ ]:
#==============================================================================
# ERROR ANALYSIS: ELASTIC NET
#==============================================================================

print("\n" + "="*80)
print("ERROR ANALYSIS: ELASTIC NET MODELS")
print("="*80)

import matplotlib.pyplot as plt

# We need to reload test data with all columns for error analysis
# (The preprocessing function only returns X matrices, not full dataframes)

print("\nReloading test data for error analysis...")
df_test_full_reload = pd.read_parquet(DATA_PATH)
df_test_full_reload['date'] = pd.to_datetime(df_test_full_reload['date'])
df_test_full_reload = df_test_full_reload[df_test_full_reload['date'] > VAL_END].copy()

print(f"Test set size: {len(df_test_full_reload):,}")

#------------------------------------------------------------------------------
# HELPER FUNCTIONS FOR ERROR ANALYSIS
#------------------------------------------------------------------------------

def compute_error_metrics(y_true, y_pred):
    """Compute RMSE, MAE, and R² for a subset."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan, 0
    y_t, y_p = y_true[mask], y_pred[mask]
    rmse = np.sqrt(mean_squared_error(y_t, y_p))
    mae = np.mean(np.abs(y_t - y_p))
    r2 = r2_score(y_t, y_p) if len(y_t) > 1 else np.nan
    return rmse, mae, r2, len(y_t)


def error_analysis_by_group(df, y_true, y_pred, group_col, group_labels=None, target_name="Target"):
    """Analyze errors by a categorical grouping variable."""
    print(f"\n{'='*60}")
    print(f"ERROR BY {group_col.upper()}: {target_name}")
    print(f"{'='*60}")

    errors = y_true - y_pred
    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['error'] = errors
    df_analysis['abs_error'] = np.abs(errors)
    df_analysis['sq_error'] = errors ** 2

    # Group and compute metrics
    if group_labels:
        df_analysis['group'] = pd.cut(df_analysis[group_col], bins=len(group_labels), labels=group_labels)
        group_col_use = 'group'
    else:
        group_col_use = group_col

    print(f"\n{'Group':<20} {'N':>10} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'R²':>10}")
    print("-"*76)

    results = []
    for group in sorted(df_analysis[group_col_use].dropna().unique()):
        mask = df_analysis[group_col_use] == group
        subset = df_analysis[mask]

        rmse = np.sqrt(subset['sq_error'].mean())
        mae = subset['abs_error'].mean()
        mean_err = subset['error'].mean()
        r2 = r2_score(subset['y_true'], subset['y_pred']) if len(subset) > 1 else np.nan

        print(f"{str(group):<20} {len(subset):>10,} {rmse:>12.6f} {mae:>12.6f} {mean_err:>+12.6f} {r2:>10.4f}")
        results.append({'group': group, 'n': len(subset), 'rmse': rmse, 'mae': mae, 'mean_error': mean_err, 'r2': r2})

    return pd.DataFrame(results)


def temporal_error_analysis(df, y_true, y_pred, date_col='date', target_name="Target"):
    """Analyze errors over time (monthly)."""
    print(f"\n{'='*60}")
    print(f"TEMPORAL ERROR ANALYSIS: {target_name}")
    print(f"{'='*60}")

    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['error'] = y_true - y_pred
    df_analysis['sq_error'] = (y_true - y_pred) ** 2
    df_analysis['abs_error'] = np.abs(y_true - y_pred)
    df_analysis['month'] = df_analysis[date_col].dt.to_period('M')

    print(f"\n{'Month':<12} {'N':>10} {'RMSE':>12} {'MAE':>12} {'Mean Error':>12} {'R²':>10}")
    print("-"*68)

    results = []
    for month in sorted(df_analysis['month'].unique()):
        mask = df_analysis['month'] == month
        subset = df_analysis[mask]

        rmse = np.sqrt(subset['sq_error'].mean())
        mae = subset['abs_error'].mean()
        mean_err = subset['error'].mean()
        r2 = r2_score(subset['y_true'], subset['y_pred']) if len(subset) > 1 else np.nan

        print(f"{str(month):<12} {len(subset):>10,} {rmse:>12.6f} {mae:>12.6f} {mean_err:>+12.6f} {r2:>10.4f}")
        results.append({'month': str(month), 'n': len(subset), 'rmse': rmse, 'mae': mae, 'mean_error': mean_err, 'r2': r2})

    return pd.DataFrame(results)


def worst_predictions(df, y_true, y_pred, n=20, target_name="Target"):
    """Show the n worst predictions."""
    print(f"\n{'='*60}")
    print(f"TOP {n} WORST PREDICTIONS: {target_name}")
    print(f"{'='*60}")

    df_analysis = df.copy()
    df_analysis['y_true'] = y_true
    df_analysis['y_pred'] = y_pred
    df_analysis['error'] = y_true - y_pred
    df_analysis['abs_error'] = np.abs(y_true - y_pred)

    # Sort by absolute error
    worst = df_analysis.nlargest(n, 'abs_error')

    # Select columns to display
    display_cols = ['cusip', 'date', 'y_true', 'y_pred', 'error', 'abs_error']

    # Add context columns if available
    for col in ['rating_num', 'ig_indicator', 'time_to_maturity', 'ytm', 'log_issue_size']:
        if col in df_analysis.columns:
            display_cols.append(col)

    print(f"\n{'CUSIP':<12} {'Date':<12} {'Actual':>10} {'Predicted':>10} {'Error':>12} {'Rating':>8} {'Maturity':>10}")
    print("-"*86)

    for _, row in worst.iterrows():
        rating = row.get('rating_num', 'N/A')
        maturity = row.get('time_to_maturity', np.nan)
        mat_str = f"{maturity:.1f}" if pd.notna(maturity) else "N/A"

        print(f"{row['cusip']:<12} {str(row['date'].date()):<12} {row['y_true']:>10.4f} {row['y_pred']:>10.4f} {row['error']:>+12.4f} {str(rating):>8} {mat_str:>10}")

    return worst


#------------------------------------------------------------------------------
# ERROR ANALYSIS FOR YTM
#------------------------------------------------------------------------------

print("\n" + "#"*80)
print("# YTM PREDICTION ERROR ANALYSIS")
print("#"*80)

# Filter for YTM samples and get predictions
df_test_ytm = df_test_full_reload.dropna(subset=['ytm']).copy()

# Recreate predictions (need to align with test set)
# Note: ytm_pred should be available from the run_elastic_net function
# If not, we need to regenerate them

# Assuming the model and scaler are still in memory
if 'ytm_model' in dir() and 'scaler_ytm' in dir():
    # Prepare features
    ytm_test_features = df_test_ytm[features_ytm].fillna(0)
    for col in features_ytm:
        ytm_test_features[col] = ytm_test_features[col].replace([np.inf, -np.inf], 0)

    X_test_ytm_analysis = scaler_ytm.transform(ytm_test_features)
    X_test_ytm_analysis = np.nan_to_num(X_test_ytm_analysis, nan=0.0, posinf=0.0, neginf=0.0)

    y_test_ytm_pred_analysis = ytm_model.predict(X_test_ytm_analysis)
    y_test_ytm_true = df_test_ytm['ytm'].values

    # 1. Error by Credit Rating
    if 'rating_num' in df_test_ytm.columns:
        # Create rating buckets
        df_test_ytm['rating_bucket'] = pd.cut(
            df_test_ytm['rating_num'],
            bins=[0, 10, 17, 22, 30],
            labels=['AAA-A (1-10)', 'BBB (11-17)', 'BB-B (18-22)', 'CCC+ and below (23+)']
        )
        error_analysis_by_group(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                                'rating_bucket', target_name="YTM")

    # Alternative: IG vs HY
    if 'ig_indicator' in df_test_ytm.columns:
        df_test_ytm['credit_category'] = df_test_ytm['ig_indicator'].map({1: 'Investment Grade', 0: 'High Yield'})
        error_analysis_by_group(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                                'credit_category', target_name="YTM")

    # 2. Error by Maturity
    if 'time_to_maturity' in df_test_ytm.columns:
        df_test_ytm['maturity_bucket'] = pd.cut(
            df_test_ytm['time_to_maturity'],
            bins=[0, 2, 5, 10, 30, 100],
            labels=['0-2Y', '2-5Y', '5-10Y', '10-30Y', '30Y+']
        )
        error_analysis_by_group(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                                'maturity_bucket', target_name="YTM")

    # 3. Error by Yield Level
    df_test_ytm['yield_bucket'] = pd.cut(
        y_test_ytm_true,
        bins=[0, 4, 6, 8, 10, 100],
        labels=['0-4%', '4-6%', '6-8%', '8-10%', '10%+']
    )
    error_analysis_by_group(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                            'yield_bucket', target_name="YTM")

    # 4. Temporal Analysis
    temporal_error_analysis(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                           target_name="YTM")

    # 5. Worst Predictions
    worst_ytm = worst_predictions(df_test_ytm, y_test_ytm_true, y_test_ytm_pred_analysis,
                                  n=20, target_name="YTM")

else:
    print("YTM model not available. Please run the YTM training section first.")


#------------------------------------------------------------------------------
# ERROR ANALYSIS FOR BID-ASK SPREAD
#------------------------------------------------------------------------------

print("\n" + "#"*80)
print("# BID-ASK SPREAD PREDICTION ERROR ANALYSIS")
print("#"*80)

# Filter for spread samples
df_test_spread = df_test_full_reload.dropna(subset=['bid_ask_spread_20d']).copy()

# Winsorize to match training
lower_bound = 0.0  # From training
upper_bound = 0.371859  # Approximate from output
df_test_spread['bid_ask_spread_20d'] = df_test_spread['bid_ask_spread_20d'].clip(lower_bound, upper_bound)

if 'spread_model' in dir() and 'scaler_spread' in dir():
    # Prepare features
    spread_test_features = df_test_spread[features_spread].fillna(0)
    for col in features_spread:
        spread_test_features[col] = spread_test_features[col].replace([np.inf, -np.inf], 0)

    X_test_spread_analysis = scaler_spread.transform(spread_test_features)
    X_test_spread_analysis = np.nan_to_num(X_test_spread_analysis, nan=0.0, posinf=0.0, neginf=0.0)

    y_test_spread_pred_analysis = spread_model.predict(X_test_spread_analysis)
    y_test_spread_true = df_test_spread['bid_ask_spread_20d'].values

    # 1. Error by Credit Rating
    if 'rating_num' in df_test_spread.columns:
        df_test_spread['rating_bucket'] = pd.cut(
            df_test_spread['rating_num'],
            bins=[0, 10, 17, 22, 30],
            labels=['AAA-A (1-10)', 'BBB (11-17)', 'BB-B (18-22)', 'CCC+ and below (23+)']
        )
        error_analysis_by_group(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                                'rating_bucket', target_name="Bid-Ask Spread")

    if 'ig_indicator' in df_test_spread.columns:
        df_test_spread['credit_category'] = df_test_spread['ig_indicator'].map({1: 'Investment Grade', 0: 'High Yield'})
        error_analysis_by_group(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                                'credit_category', target_name="Bid-Ask Spread")

    # 2. Error by Maturity
    if 'time_to_maturity' in df_test_spread.columns:
        df_test_spread['maturity_bucket'] = pd.cut(
            df_test_spread['time_to_maturity'],
            bins=[0, 2, 5, 10, 30, 100],
            labels=['0-2Y', '2-5Y', '5-10Y', '10-30Y', '30Y+']
        )
        error_analysis_by_group(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                                'maturity_bucket', target_name="Bid-Ask Spread")

    # 3. Error by Yield Level
    if 'ytm' in df_test_spread.columns:
        df_test_spread['yield_bucket'] = pd.cut(
            df_test_spread['ytm'],
            bins=[-np.inf, 4, 6, 8, 10, np.inf],
            labels=['0-4%', '4-6%', '6-8%', '8-10%', '10%+']
        )
        error_analysis_by_group(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                                'yield_bucket', target_name="Bid-Ask Spread")

    # 4. Temporal Analysis
    temporal_error_analysis(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                           target_name="Bid-Ask Spread")

    # 5. Worst Predictions
    worst_spread = worst_predictions(df_test_spread, y_test_spread_true, y_test_spread_pred_analysis,
                                     n=20, target_name="Bid-Ask Spread")

else:
    print("Spread model not available. Please run the Spread training section first.")


#------------------------------------------------------------------------------
# SUMMARY STATISTICS
#------------------------------------------------------------------------------

print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)

print("""
Key Findings to Report:

1. CREDIT RATING: Higher-rated bonds (IG) typically have lower prediction errors
   for both YTM and spread, as they have more predictable characteristics.

2. MATURITY: Very short (<2Y) and very long (>30Y) maturity bonds may show
   higher errors due to different market dynamics.

3. YIELD LEVEL: High-yield bonds (>8%) often have larger absolute errors
   due to greater idiosyncratic risk and less liquid markets.

4. TEMPORAL STABILITY: Check if errors increase in recent months, which
   could indicate model drift or changing market conditions.

5. WORST PREDICTIONS: Examine patterns in the worst predictions to identify
   systematic issues (e.g., certain sectors, recent issuances, etc.)
""")


ERROR ANALYSIS: ELASTIC NET MODELS

Reloading test data for error analysis...
Test set size: 261,527

################################################################################
# YTM PREDICTION ERROR ANALYSIS
################################################################################

ERROR BY RATING_BUCKET: YTM

Group                         N         RMSE          MAE   Mean Error         R²
----------------------------------------------------------------------------
AAA-A (1-10)            111,222     1.701075     1.426148    +1.354414    -1.5447
BB-B (18-22)                197    14.729039    10.036728    +9.881163    -0.2076
BBB (11-17)               5,792     2.662421     1.845976    +1.803007     0.1194

ERROR BY CREDIT_CATEGORY: YTM

Group                         N         RMSE          MAE   Mean Error         R²
----------------------------------------------------------------------------
High Yield              155,115    17.229441     0.859230    +0.015320     0.